# Distant Contradiction Auditor — Legal-BERT on Colab T4

Runtime → GPU T4. Upload `pairs.jsonl` to Drive (`MyDrive/dca/pairs.jsonl`) or to the session.

This notebook **only** fine-tunes and evaluates. Ingest stays on your laptop.

In [ ]:
!pip -q install transformers datasets accelerate scikit-learn

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
PAIRS = Path('/content/drive/MyDrive/dca/pairs_colab.jsonl')
gz = Path('/content/drive/MyDrive/dca/pairs_colab.jsonl.gz')
if gz.exists() and not PAIRS.exists():
    import gzip, shutil
    with gzip.open(gz, 'rb') as src, open(PAIRS, 'wb') as dst:
        shutil.copyfileobj(src, dst)
OUT = Path('/content/drive/MyDrive/dca/legalbert-dca')
assert PAIRS.exists(), 'Upload pairs.jsonl to Drive/MyDrive/dca/pairs.jsonl'
print('pairs bytes', PAIRS.stat().st_size)

In [ ]:
import json
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
import numpy as np
from sklearn.metrics import f1_score, precision_recall_fscore_support

MODEL = 'nlpaueb/legal-bert-base-uncased'
MAX_LEN = 256

rows = [json.loads(l) for l in PAIRS.read_text().splitlines() if l.strip()]
by = {}
for r in rows:
    by.setdefault(r['split'], []).append({
        'text_a': r['text_a'][:1200],
        'text_b': r['text_b'][:1200],
        'label': int(r['label']),
        'gap_bin': r.get('gap_bin', 'unknown'),
        'type': r.get('type', 'Inconsistency'),
    })
ds = DatasetDict({k: Dataset.from_list(v) for k, v in by.items()})
print(ds)

tok = AutoTokenizer.from_pretrained(MODEL)

def tokenize(batch):
    return tok(batch['text_a'], batch['text_b'], truncation=True, max_length=MAX_LEN, padding=False)

enc = ds.map(tokenize, batched=True, remove_columns=[c for c in ds['train'].column_names if c not in ('label', 'gap_bin', 'type')])
enc = enc.rename_column('label', 'labels')

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    return {'precision': p, 'recall': r, 'f1': f1}

args = TrainingArguments(
    output_dir='/content/dca-out',
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=2e-5,
    fp16=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=50,
    report_to=[],
    seed=13,
)
trainer = Trainer(model=model, args=args, train_dataset=enc['train'], eval_dataset=enc.get('dev', enc['test']), processing_class=tok, compute_metrics=compute_metrics)
trainer.train()
OUT.mkdir(parents=True, exist_ok=True)
trainer.save_model(OUT)
tok.save_pretrained(OUT)
print('saved', OUT)

In [ ]:
pred = trainer.predict(enc['test'])
logits, labels = pred.predictions, pred.label_ids
preds = np.argmax(logits, axis=-1)
p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
test_rows = by['test']
by_bin = {}
for i, row in enumerate(test_rows):
    if row['label'] != 1:
        continue
    b = row['gap_bin']
    by_bin.setdefault(b, {'yt': [], 'yp': []})
    by_bin[b]['yt'].append(1)
    by_bin[b]['yp'].append(int(preds[i]))
gap = {k: float(f1_score(v['yt'], v['yp'], zero_division=0)) for k, v in by_bin.items()}
metrics = {'model': MODEL, 'test_precision': float(p), 'test_recall': float(r), 'test_f1': float(f1), 'f1_by_gap_bin_gold_positives': gap, 'n_test': len(test_rows)}
Path('/content/drive/MyDrive/dca/metrics_legalbert.json').write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))